In [ ]:
from pathlib import Path
import os 
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from utils import get_s3
import pandas as pd 
import geopandas as gpd
import json 
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd 
from shapely.ops import linemerge, unary_union
import folium
from shapely.ops import unary_union, linemerge

In [ ]:
PATH_DATI_TRANSITI = Path(os.getcwd()) 
assert PATH_DATI_TRANSITI.exists(), f"Path {PATH_DATI_TRANSITI} does not exist"
PATH_DESCRIZIONI = PATH_DATI_TRANSITI / "descrizioni_traffico_veicolare"
PATH_DATI_STRADE = PATH_DATI_TRANSITI / 'dati_strade' 

### CLASSE 

| "CODCLASSE" | "DESCRIZIONE" |
| -- | -- |
| 1 | "Motocicli" |
| 2 | "Autovetture e monovolumi" |
| 3 | "Autovetture e monovolumi con rimorchio" |
| 4 | "Furgoni" |
| 5 | "Autocarro medio (fino a 8.7 m)" |
| 6 | "Autocarro grande (da 8.7 m)" |
| 7 | "Autocarro con rimorchio" |
| 8 | "Trattore con semirimorchio" |
| 9 | "Autobus" |

### DIREZIONI
"CODDIREZIONE" | "DESCRIZIONE"
| -- | -- |
1 | "Chilometriche crescenti"
2 | "Chilometriche decrescenti"

### VELOCITA'
|"CODVELOCITA"|"DESCRIZIONE"|
| -- | -- |
|1 |"0-20"|
|2 |"20-30"|
|3 |"30-40"|
|4 |"40-50"|
|5 |"50-60"|
|6 |"60-70"|
|7 |"70-80"|
|8 |"80-90"|
|9 |"90-100"|
|10 |"100-110"|
|11 |"110-120"|
|12 |"120-130"|
|13 |"130-140"|
|14 |">140"|

In [ ]:
dati_trasporto_veicolare = pd.read_parquet(get_s3("dati_transiti/dati_transito_veicolare.parquet"))

In [ ]:
descrizione_classi_csv = pd.read_csv(PATH_DESCRIZIONI / "descrizione_classi_veicolari.csv")
descrizione_direzioni_csv = pd.read_csv(PATH_DESCRIZIONI / "descrizione_direzioni.csv")
descrizione_velocita_csv = pd.read_csv(PATH_DESCRIZIONI / "descrizione_velocita.csv")

In [ ]:
descrizione_velocita_dict = descrizione_velocita_csv.set_index('CODVELOCITA')['DESCRIZIONE'].to_dict()
descrizione_classi_dict = descrizione_classi_csv.set_index('CODCLASSE')['DESCRIZIONE']
descrizione_direzioni_csv = descrizione_direzioni_csv.set_index('CODDIREZIONE')['DESCRIZIONE']

In [ ]:
dati_trasporto_veicolare['DESCRIZIONE_VELOCITA'] = dati_trasporto_veicolare['VELOCITA'].map(descrizione_velocita_dict)
dati_trasporto_veicolare['DESCRIZIONE_CLASSE'] = dati_trasporto_veicolare['CLASSE'].map(descrizione_classi_dict)
dati_trasporto_veicolare['DESCRIZIONE_DIREZIONE'] = dati_trasporto_veicolare['DIREZIONE'].map(descrizione_direzioni_csv)

In [ ]:
dati_trasporto_veicolare['DATA'] = pd.to_datetime(dati_trasporto_veicolare['DATA'], format='%d/%m/%Y')
dati_trasporto_veicolare['DATA_COMPLETA'] = dati_trasporto_veicolare['DATA'] + pd.to_timedelta(dati_trasporto_veicolare['ORA'].astype(int), unit='h')
dati_trasporto_veicolare

In [ ]:
# dati_trasporto_veicolare[dati_trasporto_veicolare['DATA_COMPLETA'].isna()]
dati_trasporto_veicolare

In [ ]:
dati_trasporto_veicolare.PUNTO.unique()
dati_trasporto_veicolare

Il dataframe contiene i dati di passaggi, ora per ora, per ogni classe di veicolo, direzione e velocita' di passaggio.

In [ ]:
dati_trasporto_veicolare[(dati_trasporto_veicolare['DATA'] == '2022-01-01') & (dati_trasporto_veicolare['ORA'] == 0)].groupby(['PUNTO', 'DESCRIZIONE_CLASSE', 'DESCRIZIONE_VELOCITA', 'DESCRIZIONE_DIREZIONE']).count().DATA.unique()

In [ ]:
dati_trasporto_veicolare.PUNTO.unique()

In [ ]:
viabilita = PATH_DATI_STRADE / 'viabilita'
gdf_viab= gpd.read_file(viabilita / 'viabilita_gestione_pat_v.shp')
gdf = gpd.GeoDataFrame(gdf_viab, geometry="geometry")

def plot_map_trentino(geodf, title, ax):
    geodf.plot(
        ax=ax,
        edgecolor='black',
        color = 'white',
        linewidth = 0.4
    )
    ax.set_axis_off()
    plt.title(title, fontsize=20, fontweight='bold', pad=20)
    

def create_geodf_from_geojs(geojs):
    gdf= gpd.GeoDataFrame.from_features(geojs["features"])
    gdf.set_crs(epsg=4326, inplace=True)
    gdf['random_values'] = np.random.random(len(gdf))
    return gdf

In [ ]:
lista_strade = PATH_DATI_STRADE / 'lista_strade.xls'
lista_strade_df = pd.read_excel(lista_strade, header = 1)

strade_complete = PATH_DATI_STRADE / 'strade_subset.xls'
strade_complete_df = pd.read_excel(strade_complete, header = 1)

punti_traffico = PATH_DATI_STRADE / 'punti_traffico.xls'
punti_traffico_df = pd.read_excel(punti_traffico, header = 1)

gdf_red = gdf[gdf['str_cd'].isin(strade_complete_df['id lrs'].unique())]

# gdf_red['geometry'].apply(lambda x : x.geom_type).unique()
gdf_red

In [ ]:
ss42 = gdf_red[gdf_red["str_cd"] == 20004200]
print(len(ss42))
campiglio = gdf[gdf['cod_ser'] == 'SS 239']
gdf_red.cod_ser.unique()

In [ ]:
campiglio
merged_campiglio = linemerge(campiglio['geometry'].unary_union)

lines = sorted(merged_campiglio.geoms, key=lambda l: l.coords[0][0])
merged_campiglio

In [ ]:
def plot_road_direction(gdf, ax):
    for _, group in gdf.groupby('str_cd'):
        merged = linemerge(unary_union(group.geometry))
        
        if merged.geom_type == 'MultiLineString':
            lines = sorted(merged.geoms, key=lambda l: l.coords[0][0])
            x_start, y_start = lines[0].coords[0]
            x_end, y_end = lines[-1].coords[-1]
        else:
            x_start, y_start = merged.coords[0]
            x_end, y_end = merged.coords[-1]
        
        ax.plot(x_start, y_start, marker='o', color='green', markersize=10, zorder=3)
        ax.plot(x_end, y_end, marker='o', color='red', markersize=10, zorder=3)
        ax.annotate(
            group['cod_ser'].iloc[0],
            xy=(x_start, y_start),
            xytext=(8, 8),
            textcoords="offset points",
            fontsize=8, fontweight='bold', color='darkslategray',
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=0.5, alpha=0.8),
            zorder=4
        )

In [ ]:
_, ax = plt.subplots(figsize=(18, 10))
gdf_red.plot(
    ax=ax,
    linewidth=1.5,
    color = 'purple'
    )

ax.grid(True, linestyle="--", alpha=0.5)

geojson_data_apt = json.load(get_s3( "TRENTINO-apt_2023.geojson"))
geodf_apt = create_geodf_from_geojs(geojson_data_apt)
geodf_apt = geodf_apt.to_crs(gdf_red.crs)
plot_map_trentino(geodf_apt, "Mappa viabiilta'", ax)
plot_road_direction(gdf_red, ax)

plt.tight_layout()
plt.show()

In [ ]:
# 1. Creiamo la mappa base interattiva usando il Trentino come sfondo
# Convertiamo in EPSG:4326 perché le mappe web (Folium) vogliono le coordinate geografiche
# geodf_apt_4326 = geodf_apt.to_crs(epsg=4326)
gdf_red_4326 = gdf_red.to_crs(epsg=4326)

# Inizializziamo la mappa di Folium centrata sul Trentino
# Tiles disponibili: "OpenStreetMap", "CartoDB positron" (più pulita), "CartoDB dark_matter"
# m = geodf_apt_4326.explore(
#     tiles="OpenStreetMap", 
#     style_kwds={'fillColor': 'white', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.3},
#     name="Confini Trentino (APT)"
# )

# 2. Aggiungiamo le strade in viola
m =gdf_red_4326.explore(
    # m=m, # Specifichiamo di disegnarle sulla mappa 'm' appena creata
    color="purple",
    style_kwds={'weight': 3},
    name="Strade"
)

# 3. Calcoliamo e aggiungiamo i versi delle strade (Inizio = Verde, Fine = Rosso)
# Creiamo due FeatureGroup per poter accendere/spegnere i marker dalla legenda
fg_inizio = folium.FeatureGroup(name="Inizio Strada (Verde)").add_to(m)
fg_fine = folium.FeatureGroup(name="Fine Strada (Rosso)").add_to(m)

for _, group in gdf_red_4326.groupby('str_cd'):
    merged = linemerge(unary_union(group.geometry))
    
    # Stessa logica del tuo codice per estrarre inizio e fine
    if merged.geom_type == 'MultiLineString':
        lines = sorted(merged.geoms, key=lambda l: l.coords[0][0])
        x_start, y_start = lines[0].coords[0]
        x_end, y_end = lines[-1].coords[-1]
    else:
        x_start, y_start = merged.coords[0]
        x_end, y_end = merged.coords[-1]
    
    # ATTENZIONE: Folium vuole le coordinate in formato (Latitudine, Longitudine) -> (y, x)
    cod_ser = group['cod_ser'].iloc[0]
    
    # Marker di Inizio (Verde) con popup interattivo al click
    folium.CircleMarker(
        location=(y_start, x_start),
        radius=6,
        color="green",
        fill=True,
        fill_color="green",
        fill_opacity=0.8,
        popup=f"<b>Cod. Servizio:</b> {cod_ser}<br><b>Stato:</b> INIZIO"
    ).add_to(fg_inizio)
    
    # Marker di Fine (Rosso)
    folium.CircleMarker(
        location=(y_end, x_end),
        radius=6,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.8,
        popup=f"<b>Cod. Servizio:</b> {cod_ser}<br><b>Stato:</b> FINE"
    ).add_to(fg_fine)

# Aggiungiamo il controllo dei livelli (per accendere/spegnere strade o punti)
folium.LayerControl().add_to(m)
m.save("mappa_viabilita_trentino.html")

## Appendix (and more inspections)

In [ ]:
lista_strade_df['id LRS'].unique()
lista_strade_df = lista_strade_df.rename(columns = {'id LRS':'id_lrs'})
lista_strade_df

In [ ]:
strade_complete_df['id lrs'].unique()
strade_complete_df = strade_complete_df.rename(columns = {'id lrs':'id_lrs'})
strade_complete_df

In [ ]:
punti_traffico_df = punti_traffico_df.rename(columns = {'LOCALITA\'': 'LOCALITA'})
punti_traffico_df

In [ ]:
# --- 1. SPOSTIAMO LE INFO DI INIZIO STRADA IN UN DIZIONARIO ---
km_inizio_strade = dict(zip(strade_complete_df['codice servizio'], strade_complete_df['progressiva di inizio strada']))

gdf_strade_unite = gdf_red.dissolve(by='cod_ser')  # raggruppare per 'cod_ser' (es. SS 42)
gdf_strade_unite['geometry'] = gdf_strade_unite['geometry'].apply(linemerge)  # linemerge unisce i segmenti adiacenti in un'unica LineString continua

def compute_loc_spira(row, gdf_strade, km_inizio_dict):
    strada = row['STRADA']  
    km_spira = row['KM']        
    
    if strada in gdf_strade.index:
        geom_strada = gdf_strade.loc[strada, 'geometry']
        km_inizio_strada = km_inizio_dict.get(strada, 0.0)
        
        dist_spira = km_spira - km_inizio_strada
        distanza_metri = dist_spira * 1000 # Convertiamo in metri 
        
        if distanza_metri >= 0 and distanza_metri <= geom_strada.length:
            return geom_strada.interpolate(distanza_metri)
    return None


In [ ]:
punti_traffico_df['geometry'] = punti_traffico_df.apply(
    lambda row: compute_loc_spira(row, gdf_strade_unite, km_inizio_strade), 
    axis=1
)
punti_traffico_df = gpd.GeoDataFrame(punti_traffico_df, geometry='geometry')
punti_traffico_df.set_crs(gdf_red.crs, inplace=True)

In [ ]:
# --- 5. PLOTTING FINALE (IL TUO CODICE AGGIORNATO) ---
_, ax = plt.subplots(figsize=(18, 10))

gdf_red.plot(
    ax=ax,
    linewidth=1.5,
    color='purple',
    label='Strade Analizzate'
)

# SEZIONE MAPPA TRENTINO APT (Tuo codice esistente)
geojson_data_apt = json.load(get_s3("TRENTINO-apt_2023.geojson"))
geodf_apt = create_geodf_from_geojs(geojson_data_apt)
geodf_apt = geodf_apt.to_crs(gdf_red.crs)
plot_map_trentino(geodf_apt, "Mappa Viabilità e Posizione Spire", ax)

# !!! NUOVO: Plot delle Spire come punti rossi  sullamappa !!!
punti_traffico_df.plot(
    ax=ax,
    color='red',
    marker='o',
    markersize=120,          # Dimensione del pallino
    edgecolor='black',       # Contorno per renderle visibili
    linewidth=1,
    label='Spire di Rilevamento',
    zorder=5                 # Assicura che i punti stiano SOPRA le linee
)

# Aggiunge le etichette con il nome della località vicino alla spira (Opzionale ma utile)
for idx, row in punti_traffico_df.iterrows():
    if row['geometry'] is not None:
        ax.annotate(
            text=row['LOCALITA'], 
            xy=(row['geometry'].x, row['geometry'].y),
            xytext=(8, 8), 
            textcoords="offset points", 
            fontsize=9, 
            fontweight='bold',
            color='darkred'
        )

ax.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc='upper left', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
gdf_strade_web = gdf_red.to_crs(epsg=4326)
gdf_spire_web = punti_traffico_df.to_crs(epsg=4326)

m = gdf_strade_web.explore(
    column='nome',             
    cmap='Set1',             
    style_kwds=dict(weight=4),
    tooltip=['nome', 'cod_ser'], # Cosa vedi quando passi il mouse sopra la strada
    popup=True,              
    name="Rete Stradale"       
)

# 3. Aggiungiamo le SPIRE sulla STESSA mappa
gdf_spire_web.explore(
    m=m,                     
    color='violet',               # Colore dei pallini
    marker_kwds=dict(
        radius=7, 
        fill=True,
        color='black',         # Bordo del pallino
        weight=1
    ),
    tooltip='LOCALITA',       
    popup=['STRADA', 'KM'],
    name="Spire di Rilevamento"
)

folium.LayerControl().add_to(m)
m.save("mappa_interattiva_spire.html")